# Week 26: Cloud-native: COG, Zarr, STAC catalogs

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/26/](https://launchdetect.com/academy/week/26/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_The old way: download a 10 GB GeoTIFF. The new way: range-request just the bytes you need from S3. COG, Zarr, and STAC make this possible._


## Why this week matters

Cloud-native formats are the difference between downloading a 5 GB Landsat scene and range-requesting the 50 KB you actually need. Production geospatial in 2026 means COG + Zarr + STAC. No exceptions.


## Learning objectives

By the end of this lab you will be able to:

- Understand COG (Cloud-Optimized GeoTIFF) structure
- Use Zarr for multi-dimensional gridded data
- Build and query a STAC catalog
- Range-request a tile out of a COG without downloading the file


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Query Microsoft Planetary Computer's STAC catalog for Sentinel-2 scenes over Cape Canaveral. The query returns COG asset URLs you can read partial bytes from with rio-tiler.


In [ ]:
# Week 26: query the Microsoft Planetary Computer STAC catalog.
# Range-request a single tile from a COG without downloading the full scene.

try:
    from pystac_client import Client
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")
    search = cat.search(
        collections=["sentinel-2-l2a"],
        bbox=[-80.7, 28.5, -80.5, 28.7],   # Cape Canaveral
        datetime="2024-01-01/2024-03-01",
        max_items=5,
    )
    items = list(search.items())
    print(f"Found {len(items)} Sentinel-2 scenes")
    for item in items[:3]:
        print(f"  {item.id}  cloud={item.properties.get('eo:cloud_cover', '?'):.1f}%")
except Exception as e:
    print(f"STAC query failed (likely no network in Colab init): {e}")

# TODO: use rio-tiler to range-request a single 256x256 tile out of the cloudiest
# scene's COG asset, time it, compare against downloading the full file.


## What just happened — and why it works

STAC is a JSON spec for describing geospatial assets. Catalog → Collection → Item → Asset (the COG URL). A search by bbox + datetime returns matching items in milliseconds, even from a catalog of millions.


## Common gotchas

- Microsoft Planetary Computer is free but has rate limits. Cache results.
- AWS Earth Search returns Sentinel-2 from a different bucket than the original Copernicus distribution. Check the asset URL.
- COG range requests need HTTPS, not HTTP. Some legacy mirrors don't support range requests at all.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/26/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
